In [ ]:
# tom99763 (rank 8, LB 0.954), 2026-07-11, thread "beware of jumps in ground truth track":
#   "Videos sharing the schedule / Frames duplicated after index...
#    6bba_05b6850b - 07477033 - 5b28472a : 4, 12, 27, 42, 52, 57, 59, 62, 66, 76"
#
# 6bba_05b6850b is one of OUR four verification clips. If frames really are duplicated,
# then across those t -> t+1 pairs the true displacement is exactly zero, which is the same
# phenomenon notes/59 measured from the other side: 8.4% of GT links have zero displacement.
# Nothing in the public notebook lineage mentions it, and an identity link on a duplicated
# frame pair is exact by construction.
#
# This checks the pixels, not the claim.
import json, subprocess, sys
from pathlib import Path
import numpy as np
try:
    import zarr
except ModuleNotFoundError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "zarr>=3"], check=True)
    import zarr

COMP = Path("/kaggle/input/competitions/biohub-cell-tracking-during-development")
CLAIM = {"6bba_05b6850b": [4, 12, 27, 42, 52, 57, 59, 62, 66, 76]}

def dup_frames(stem, folder="train"):
    a = zarr.open_array(str(COMP / folder / f"{stem}.zarr" / "0"), mode="r")
    T = a.shape[0]
    prev = np.asarray(a[0]); dups = []
    for t in range(1, T):
        cur = np.asarray(a[t])
        if np.array_equal(prev, cur):
            dups.append(t)
        prev = cur
    return T, dups

for stem, claimed in CLAIM.items():
    T, dups = dup_frames(stem)
    print(f"{stem}: T={T}")
    print(f"  duplicated frames (identical to t-1): {dups}")
    print(f"  tom99763 claimed after-index:          {claimed}")
    print(f"  ours-minus-one:                        {[d-1 for d in dups]}")
    print(f"  fraction of frame pairs duplicated:    {len(dups)/(T-1):.1%}")

In [ ]:
# If it holds for one clip, how common is it across the whole training set? Sampling 40
# movies is enough to say whether this is a quirk of one video or a property of the data.
import random
stems = sorted(p.name.split(".")[0] for p in (COMP / "train").glob("*.zarr"))
random.seed(0)
sample = stems[:5] + random.sample(stems, 35)
tot_pairs = tot_dups = 0
per = []
for s in sample:
    try:
        T, d = dup_frames(s)
    except Exception as e:
        print(f"{s}: {type(e).__name__}"); continue
    per.append((s, T, len(d), d[:8]))
    tot_pairs += T - 1; tot_dups += len(d)
per.sort(key=lambda x: -x[2])
print(f"{'stem':<18}{'T':>5}{'dups':>6}  first few")
for s, T, n, d in per:
    print(f"{s:<18}{T:>5}{n:>6}  {d}")
print(f"\nTOTAL {tot_dups} duplicated pairs of {tot_pairs} ({tot_dups/max(tot_pairs,1):.2%})")
print(f"movies with at least one duplicate: {sum(1 for _,_,n,_ in per if n)} of {len(per)}")